# Natural Attack Angle (NAA) — Frontier League
Derives each batter's Natural Attack Angle by fitting a weighted quadratic to the
launch angle vs. exit velocity envelope, then finding the vertex.

**Method**
1. Filter to in-play batted ball events
2. Bin launch angles into 15 equal-width bins (−30° to 60°)
3. Flag the top EV contacts in each bin (top 20% or 3, whichever is smaller)
4. Fit a weighted quadratic to those envelope points (nearest-neighbor weights)
5. Vertex of the curve = Natural Attack Angle + Max EV

**Output**: `naa_leaderboard.csv` with one row per batter


In [8]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import huber
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports OK")


Imports OK


## 1. Config & Data Load

In [9]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH      = "../2025_fl_tagged_cleaned/25_ALL_GAMES_cleaned.csv"
MIN_BBE        = 20          # minimum batted ball events to appear on leaderboard

ANGLE_MIN      = -30         # degrees — left edge of fitting window
ANGLE_MAX      =  60         # degrees — right edge of fitting window
N_BINS         =  15         # number of launch angle bins
CURVE_MIN      = -60         # degrees — left edge of plot/vertex search
CURVE_MAX      =  60         # degrees — right edge
PRECISION      =  4          # points per degree for vertex search
CURVE_POINTS   = (CURVE_MAX - CURVE_MIN) * PRECISION + 1

# ── Load ──────────────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw rows: {len(raw):,}")
print(f"Columns: {list(raw.columns[:15])} ...")


Raw rows: 247,418
Columns: ['PitchNo', 'Date', 'Time', 'PAofInning', 'PitchofPA', 'Pitcher', 'PitcherId', 'PitcherThrows', 'PitcherTeam', 'Batter', 'BatterId', 'BatterSide', 'BatterTeam', 'PitcherSet', 'Inning'] ...


## 2. Filter to Batted Ball Events

In [10]:
inplay = (
    raw[raw["PitchCall"] == "InPlay"]
    .drop_duplicates(subset=["Date", "Time", "Pitcher", "Batter"])
    .dropna(subset=["Angle", "ExitSpeed"])
    .reset_index(drop=True)
)
inplay["Angle"]     = pd.to_numeric(inplay["Angle"],     errors="coerce")
inplay["ExitSpeed"] = pd.to_numeric(inplay["ExitSpeed"], errors="coerce")
inplay = inplay.dropna(subset=["Angle", "ExitSpeed"]).reset_index(drop=True)

print(f"InPlay BBE: {len(inplay):,}")
print(f"Unique batters: {inplay['Batter'].nunique()}")


InPlay BBE: 42,139
Unique batters: 394


## 3. Bin Launch Angles & Flag Top EV Contacts

In [11]:
def bin_and_flag(df: pd.DataFrame) -> pd.DataFrame:
    """
    For a single batter's BBE dataframe:
    - Bins Angle into N_BINS equal-width bins from ANGLE_MIN to ANGLE_MAX
    - Flags the top EV contacts in each bin (top 20% or 3, whichever is smaller)
    """
    df = df.copy()

    if df["Angle"].nunique() <= 1:
        df["LA_Bin"]        = "1"
        df["top_ev_in_bin"] = 0
        return df

    bin_edges  = np.linspace(ANGLE_MIN, ANGLE_MAX, N_BINS + 1)
    bin_labels = [str(i) for i in range(1, N_BINS + 1)]

    df["LA_Bin"] = pd.cut(
        df["Angle"],
        bins=bin_edges,
        labels=bin_labels,
        include_lowest=True,
    )
    df["top_ev_in_bin"] = 0

    for label in bin_labels:
        bin_rows = df[df["LA_Bin"] == label]
        if bin_rows.empty:
            continue
        k = min(3, math.ceil(0.20 * len(bin_rows)))
        top_idx = bin_rows["ExitSpeed"].nlargest(k).index
        df.loc[top_idx, "top_ev_in_bin"] = 1

    return df


# Apply per batter
batter_dfs = {}
for batter, grp in inplay.groupby("Batter"):
    batter_dfs[batter] = bin_and_flag(grp)

print(f"Processed {len(batter_dfs)} batters")

# Quick sanity check on one batter
sample_batter = list(batter_dfs.keys())[0]
print(f"\nSample ({sample_batter}):")
print(batter_dfs[sample_batter][["Angle", "ExitSpeed", "LA_Bin", "top_ev_in_bin"]].head(8))


Processed 394 batters

Sample (Abbott, Sam):
       Angle  ExitSpeed LA_Bin  top_ev_in_bin
8106  55.152     77.978     15              1
8119 -65.912     78.336    NaN              0
8484  73.090     79.832    NaN              0
8504  -9.580     68.937      4              1
8902  34.346     74.785     11              0
8913  45.935     93.360     13              0
9258  10.386     93.705      7              0
9269 -44.210     43.615    NaN              0


## 4. Curve Fitting Functions

In [12]:
def quadratic(x, a, b, c):
    return a * x**2 + b * x + c


def nn_weights(x_data: np.ndarray, n_neighbors: int = 4) -> np.ndarray:
    """Inverse-distance weights based on the n-th nearest neighbor distance."""
    if len(x_data) < n_neighbors + 1:
        return np.ones(len(x_data))
    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(x_data.reshape(-1, 1))
    distances, _ = nbrs.kneighbors(x_data.reshape(-1, 1))
    return 1.0 / (distances[:, n_neighbors] + 1e-8)


def cost_weighted(params, x, y):
    a, b, c = params
    residuals = y - quadratic(x, a, b, c)
    weights   = nn_weights(x)
    return np.mean(weights * residuals**2)


def cost_huber(params, x, y):
    a, b, c = params
    residuals    = y - quadratic(x, a, b, c)
    abs_res      = np.abs(residuals)
    delta        = np.percentile(abs_res, 95)
    return np.mean(huber(delta, residuals))


def fit_curve(x: np.ndarray, y: np.ndarray, method: str = "weighted"):
    """
    Fit a quadratic to (x, y) using 'weighted' or 'huber' cost.
    Returns (a, b, c) or (nan, nan, nan) if fitting fails.
    """
    if len(x) < 3:
        return np.array([np.nan, np.nan, np.nan])
    cost_fn = cost_weighted if method == "weighted" else cost_huber
    res = minimize(cost_fn, x0=[1.0, 0.0, 1.0], args=(x, y))
    return res.x if res.success else np.array([np.nan, np.nan, np.nan])


print("Curve fitting functions defined")


Curve fitting functions defined


## 5. Extract Natural Attack Angle (Vertex of Fitted Curve)

In [13]:
def get_naa(batter_df: pd.DataFrame, method: str = "weighted"):
    """
    Given a single batter's binned BBE dataframe, fits the EV envelope curve
    and returns (natural_attack_angle, max_ev).
    """
    envelope = batter_df[
        (batter_df["top_ev_in_bin"] == 1) &
        (batter_df["Angle"] >= ANGLE_MIN) &
        (batter_df["Angle"] <= ANGLE_MAX)
    ]

    x = envelope["Angle"].values.astype(float)
    y = envelope["ExitSpeed"].values.astype(float)

    if len(x) < 3:
        return np.nan, np.nan

    coeffs = fit_curve(x, y, method=method)
    if np.isnan(coeffs).any():
        return np.nan, np.nan

    # Search for vertex over a dense grid
    x_grid = np.linspace(CURVE_MIN, CURVE_MAX, CURVE_POINTS)
    y_grid  = quadratic(x_grid, *coeffs)

    naa    = x_grid[np.argmax(y_grid)]
    max_ev = np.max(y_grid)
    return float(naa), float(max_ev)


print("get_naa defined")


get_naa defined


## 6. Build Leaderboard

In [14]:
records = []
for batter, df in batter_dfs.items():
    bbe  = len(df)
    team = df["BatterTeam"].iloc[0] if "BatterTeam" in df.columns else np.nan
    side = df["BatterSide"].iloc[0] if "BatterSide" in df.columns else np.nan

    naa, max_ev = get_naa(df, method="weighted")

    records.append({
        "batter":    batter,
        "team":      team,
        "side":      side,
        "bbe":       bbe,
        "naa":       naa,
        "max_ev":    max_ev,
    })

leaderboard = (
    pd.DataFrame(records)
    .sort_values("bbe", ascending=False)
    .reset_index(drop=True)
)

qualified = leaderboard[leaderboard["bbe"] >= MIN_BBE].copy()

print(f"Total batters: {len(leaderboard)}")
print(f"Qualified (≥{MIN_BBE} BBE): {len(qualified)}")
print(f"\nNAA missingness: {leaderboard['naa'].isna().mean():.1%}")
print(f"\nNAA distribution (qualified):")
print(qualified["naa"].describe().round(2))


KeyboardInterrupt: 

## 7. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# NAA distribution
axes[0].hist(qualified["naa"].dropna(), bins=30,
             color="#1565C0", alpha=0.85, edgecolor="white", linewidth=0.3)
axes[0].axvline(qualified["naa"].mean(), color="red", linewidth=1.5,
                linestyle="--", label=f"Mean = {qualified['naa'].mean():.1f}°")
axes[0].set_xlabel("Natural Attack Angle (°)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"NAA Distribution  (n={len(qualified)} qualified batters)")
axes[0].legend(fontsize=9)

# Max EV distribution
axes[1].hist(qualified["max_ev"].dropna(), bins=30,
             color="#1565C0", alpha=0.85, edgecolor="white", linewidth=0.3)
axes[1].axvline(qualified["max_ev"].mean(), color="red", linewidth=1.5,
                linestyle="--", label=f"Mean = {qualified['max_ev'].mean():.1f} mph")
axes[1].set_xlabel("Max EV from curve (mph)")
axes[1].set_ylabel("Count")
axes[1].set_title("Max EV Distribution")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# NAA vs Max EV scatter
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(qualified["naa"], qualified["max_ev"],
           alpha=0.5, s=qualified["bbe"] / 3, color="#1565C0")
ax.set_xlabel("Natural Attack Angle (°)")
ax.set_ylabel("Max EV (mph)")
ax.set_title("NAA vs Max EV  (point size = BBE)")
plt.tight_layout()
plt.show()


## 8. Plot Individual Batter Curve
Change `BATTER` to any name in the dataset.

In [ ]:
def plot_naa_curve(batter_name: str, method: str = "weighted"):
    if batter_name not in batter_dfs:
        print(f"{batter_name} not found. Available names sample:")
        print(list(batter_dfs.keys())[:10])
        return

    df     = batter_dfs[batter_name]
    naa, max_ev = get_naa(df, method=method)

    envelope = df[df["top_ev_in_bin"] == 1]
    rest     = df[df["top_ev_in_bin"] == 0]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(rest["Angle"],     rest["ExitSpeed"],     alpha=0.2, s=10,
               color="gray",     label="All contacts")
    ax.scatter(envelope["Angle"], envelope["ExitSpeed"], alpha=0.8, s=25,
               color="#1565C0",  label="Envelope (top EV per bin)")

    # Fitted curve
    x_env = envelope["Angle"].values.astype(float)
    y_env = envelope["ExitSpeed"].values.astype(float)
    if len(x_env) >= 3:
        coeffs  = fit_curve(x_env, y_env, method=method)
        if not np.isnan(coeffs).any():
            x_grid = np.linspace(CURVE_MIN, CURVE_MAX, CURVE_POINTS)
            y_grid  = quadratic(x_grid, *coeffs)
            ax.plot(x_grid, y_grid, color="red", linewidth=2, label="Fitted curve")
            if not np.isnan(naa):
                ax.axvline(naa, color="red", linewidth=1, linestyle="--",
                           label=f"NAA = {naa:.1f}°")

    ax.set_xlim(-80, 80)
    ax.set_ylim(0, 120)
    ax.set_xlabel("Launch Angle (°)")
    ax.set_ylabel("Exit Velocity (mph)")
    ax.set_title(f"{batter_name}  —  NAA = {naa:.1f}°,  Max EV = {max_ev:.1f} mph")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


BATTER = "Sandle, Michael"   # ← change me
plot_naa_curve(BATTER)


## 9. Save

In [ ]:
out_path = OUTPUT_DIR / "naa_leaderboard.csv"
leaderboard.to_csv(out_path, index=False)
print(f"Saved {len(leaderboard)} rows → {out_path}")
print(f"\nTop 20 by NAA (qualified):")
print(
    qualified.dropna(subset=["naa"])
    .sort_values("naa", ascending=False)
    [["batter", "team", "side", "bbe", "naa", "max_ev"]]
    .head(20)
    .to_string(index=False)
)
